In [1]:
# 必要なモジュールをインポート
import os
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd

# 環境変数の取得
load_dotenv()

# OpenAI APIクライアントを生成
client = OpenAI(api_key=os.environ['API_KEY'])

# モデル名
MODEL_NAME = "gpt-4o-mini"

In [2]:
# 1. Excelファイルを読み込む
df = pd.read_excel('サンプルデータ.xlsx', sheet_name='売上データ')
# データフレームを表示して確認
df.head()

,カテゴリー,商品コード,商品名,売上日,単価,数量,原価
0,食品,1001,りんご,2023-01-01,200,50,120
1,食品,1002,バナナ,2023-01-01,150,100,80
2,食品,1003,牛乳,2023-01-02,180,80,100
3,衣服,2001,Tシャツ,2023-01-02,1500,20,800
4,衣服,2002,ジーンズ,2023-01-03,5000,10,2500


In [3]:
# 2. データをLLM用にテキスト形式に変換
# データフレーム全体を文字列に変換
sales_data_text = df.astype(str)
prompt_text = f"売上データ:\n{sales_data_text}\nこの売上データの傾向を分析してください。"
# 表示して確認
print(prompt_text)

売上データ:
    カテゴリー 商品コード      商品名         売上日    単価   数量    原価
0      食品  1001      りんご  2023-01-01   200   50   120
1      食品  1002      バナナ  2023-01-01   150  100    80
2      食品  1003       牛乳  2023-01-02   180   80   100
3      衣服  2001     Tシャツ  2023-01-02  1500   20   800
4      衣服  2002     ジーンズ  2023-01-03  5000   10  2500
..    ...   ...      ...         ...   ...  ...   ...
235    衣服  2077   レインパンツ  2023-04-28  2000   18  1000
236    食品  1085      ザクロ  2023-04-29   600   40   300
237   日用品  3077    バスブラシ  2023-04-29   400   60   200
238    衣服  2078  レインシューズ  2023-04-30  2500   15  1250
239    食品  1086    ココナッツ  2023-04-30   300   80   150

[240 rows x 7 columns]
この売上データの傾向を分析してください。


In [4]:
# 3. OpenAI APIの呼び出し

# 役割を設定
role = "あなたはマーケティング分野に精通したデータサイエンティストです。企業の成長をサポートするために、効果的なインサイトを提供します。"

# APIへリクエスト
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": role},
        {"role": "user", "content": prompt_text},
    ],
)

# LLMからの回答を表示
print(response.choices[0].message.content.strip())

売上データの分析を行う際に、以下のポイントを考察しましょう。主な焦点を以下に示します。

### 1. カテゴリー別の売上分析
- **カテゴリーごとの売上合計**: 各カテゴリー（食品、衣服、日用品など）の売上合計を求め、どのカテゴリーが最も収益を上げているかを評価します。
- **利益率**: 各カテゴリー内での単価と原価を基に、利益率を計算し、どのカテゴリーが最も高い利益を生んでいるかを理解します。

### 2. 商品別の人気分析
- **売上上位の商品**: 売上が最も高かった商品を特定し、これらの商品の特徴（価格、数量など）を分析します。
- **売上トレンド**: 売上日別のデータを使用して、異なる商品の売上トレンドを可視化します（例: 月ごとの合計売上など）。

### 3. 季節性と時間の影響
- **季節性の確認**: 売上の月別トレンドを分析することで、年間を通じた季節性を確認し、特定のシーズンに売上が増加する商品を特定します。
- **曜日や祭日ごとの売上**: 売上日を基に、特定の曜日や祝日による影響を分析します。

### 4. 顧客セグメンテーション
- **購入パターンの分析**: 特定のカテゴリーや商品を好んで購入する顧客セグメントを特定し、その特性を分析します。

### 5. その他の考察
- **価格設定の効果**: 単価の変動と売上変動の相関を分析し、価格戦略の調整に役立てる情報を得ます。
- **販売促進活動**: 特定の売上増加が販売促進活動（プロモーションなど）に関連しているかなどの因果関係を探ることも有益です。

### 結論
データをもとに、売上のトレンドや商品ごとの特色、カテゴリー間の競争力を分析することで、次のマーケティング戦略を練るためのインサイトが得られます。具体的な数値や可視化がないため、上記のアプローチをもとに分析を行い、結果を導き出していただければと思います。必要に応じて、具体的なフィルタリングや追加の計算を行うことが求められるでしょう。


In [5]:
# 4. 分析結果をデータフレームに変換
result_list = response.choices[0].message.content.strip().split("\n")
df_out = pd.DataFrame(result_list, columns=['結果'])
print(df_out)

                                                   結果
0         売上データの分析を行う際に、以下のポイントを考察しましょう。主な焦点を以下に示します。
1                                                    
2                                  ### 1. カテゴリー別の売上分析
3   - **カテゴリーごとの売上合計**: 各カテゴリー（食品、衣服、日用品など）の売上合計を求...
4   - **利益率**: 各カテゴリー内での単価と原価を基に、利益率を計算し、どのカテゴリーが最...
5                                                    
6                                     ### 2. 商品別の人気分析
7   - **売上上位の商品**: 売上が最も高かった商品を特定し、これらの商品の特徴（価格、数量...
8   - **売上トレンド**: 売上日別のデータを使用して、異なる商品の売上トレンドを可視化しま...
9                                                    
10                                   ### 3. 季節性と時間の影響
11  - **季節性の確認**: 売上の月別トレンドを分析することで、年間を通じた季節性を確認し、...
12      - **曜日や祭日ごとの売上**: 売上日を基に、特定の曜日や祝日による影響を分析します。
13                                                   
14                                 ### 4. 顧客セグメンテーション
15  - **購入パターンの分析**: 特定のカテゴリーや商品を好んで購入する顧客セグメントを特定...
16                                                   
17                          

In [6]:
# 5. 結果をExcelファイルに保存
df_out.to_excel("売上データ分析結果.xlsx", index=False)

In [7]:
# ワークフロー化
print("処理を開始します。")

# 1. Excelファイルを読み込む
df = pd.read_excel('サンプルデータ.xlsx', sheet_name='売上データ')

# 2. データをLLM用にテキスト形式に変換
sales_data_text = df.astype(str)
prompt_text = f"売上データ:\n{sales_data_text}\nこの売上データの傾向を分析してください。"

# 3. OpenAI APIの呼び出し
# 役割を設定
role = "あなたはマーケティング分野に精通したデータサイエンティストです。企業の成長をサポートするために、効果的なインサイトを提供します。"
# APIへリクエスト
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": role},
        {"role": "user", "content": prompt_text},
    ],
)

# 4. 分析結果をデータフレームに変換
result_list = response.choices[0].message.content.strip().split("\n")
df_out = pd.DataFrame(result_list, columns=['結果'])

# 5. 結果をExcelファイルに保存
df_out.to_excel("売上データ分析結果.xlsx", index=False)

print("Excelファイルに分析結果を保存しました。")

処理を開始します。
Excelファイルに分析結果を保存しました。
